## はじめにこのノートブックでは、Part 1 で作った Gold 層テーブルと `setup.sql` のキュレーション済テーブルを**セマンティックビュー**にまとめ、Cortex Analyst で自然言語から SQL を生成できる状態にします。**前提条件:**- `part1_ai_functions.ipynb` が実行済みであること（`GOLD_COMPANY_NEWS_ANALYZED` が必要）- ウェアハウスに `SNOW_AM_WH` が選択されていること**主な処理内容:**- Marketplace の生データがそのままでは使えない理由の確認- メタデータの自動生成（`AI_GENERATE_TABLE_DESC`）- セマンティックビューの作成（`CREATE SEMANTIC VIEW`）- `SEMANTIC_VIEW()` 構文による動作確認- Cortex Analyst への自然言語質問（Snowsight UI と REST API）**作成されるオブジェクト:**| オブジェクト | 内容 ||---|---|| `PORTFOLIO_MARKET_SEMANTIC_VIEW` | 保有明細・株価・財務諸表・ニュースを統合したセマンティックビュー |> ⏱️ **このパートの目安時間: 30分**

In [ ]:
-- ==========================================================================-- 環境設定-- ==========================================================================USE DATABASE SNOW_AM_DB;USE SCHEMA MARKET_INTELLIGENCE;USE WAREHOUSE SNOW_AM_WH;-- Part 1 の成果物が揃っているか確認しますSELECT 'GOLD_COMPANY_NEWS_ANALYZED' AS "テーブル", COUNT(*) AS "件数" FROM GOLD_COMPANY_NEWS_ANALYZEDUNION ALL SELECT 'GOLD_EARNINGS_CALL_CHUNKS', COUNT(*) FROM GOLD_EARNINGS_CALL_CHUNKSUNION ALL SELECT 'FACT_FUND_HOLDING',         COUNT(*) FROM FACT_FUND_HOLDINGUNION ALL SELECT 'FACT_STOCK_PRICE_DAILY',    COUNT(*) FROM FACT_STOCK_PRICE_DAILYUNION ALL SELECT 'FACT_FINANCIAL_METRICS',    COUNT(*) FROM FACT_FINANCIAL_METRICSORDER BY 1;

## 1. なぜセマンティックビューが必要かAI に「NVDAの直近の終値を教えて」と聞いたとき、AI はテーブル名とカラム名から推測して SQL を書きます。推測が当たれば正しい答えが返りますが、外れると**もっともらしい間違った SQL**が返ってきます。セマンティックビューは、この推測を排除するための仕組みです。

### 1-1. Marketplace の生データはそのままでは使えないまず、Marketplace から取得した株価データの生の形を見てみましょう。

In [ ]:
-- ==========================================================================-- Marketplace の生データ（縦持ち）-- ==========================================================================-- NVDA の「1営業日分」のデータです。何行返ってくるでしょうか？SELECT TICKER        AS "銘柄",       DATE          AS "日付",       VARIABLE      AS "変数名",       VARIABLE_NAME AS "変数の説明",       VALUE         AS "値"FROM SNOWFLAKE_PUBLIC_DATA.PUBLIC_DATA.STOCK_PRICE_TIMESERIESWHERE TICKER = 'NVDA'  AND DATE = (SELECT MAX(DATE)              FROM SNOWFLAKE_PUBLIC_DATA.PUBLIC_DATA.STOCK_PRICE_TIMESERIES              WHERE TICKER = 'NVDA')ORDER BY VARIABLE;

In [ ]:
-- ==========================================================================-- キュレーション後（横持ち）-- ==========================================================================-- setup.sql の Step 6 でピボットしたテーブルですSELECT TICKER      AS "銘柄",       TRADE_DATE  AS "取引日",       OPEN_PRICE  AS "始値",       HIGH_PRICE  AS "高値",       LOW_PRICE   AS "安値",       CLOSE_PRICE AS "終値",       VOLUME      AS "出来高"FROM FACT_STOCK_PRICE_DAILYWHERE TICKER = 'NVDA'ORDER BY TRADE_DATE DESCLIMIT 5;

> **💡 比較のポイント**>> 同じ「1営業日の NVDA の株価」が、生データでは **9行**、キュレーション後では **1行**です。>> 生データは `VARIABLE` / `VALUE` の**縦持ち**構造です。この形のまま Cortex Analyst に渡すと、> AI は「終値を取るには `WHERE VARIABLE = 'post-market_close'` が必要」ということを> 知る手段がありません。結果として `VALUE` を無条件に集計するなど、静かに間違った SQL を書きます。>> Marketplace のデータは「そのまま使える」のではなく、**分析可能な形に整える工程が必要**です。> `setup.sql` の Step 6 でやったことがまさにそれで、これは AI 活用の前提条件になります。

### 1-2. セマンティックビューが加える3つの情報横持ちに整えただけでは、まだ足りません。セマンティックビューは次の3つを AI に教えます。| 加える情報 | ないと起きること | セマンティックビューでの表現 ||---|---|---|| **テーブル同士の結合条件** | 誤った JOIN や、そもそも JOIN されない | `RELATIONSHIPS` || **業務指標の定義** | 「AUM」と言われても計算式がわからない | `METRICS` || **業務用語との対応** | 「運用資産残高」で検索してもヒットしない | `WITH SYNONYMS` |セマンティックビューは以下の5つのブロックで構成されます。```TABLES        …… どのテーブルを使うか。主キーと同義語を定義するRELATIONSHIPS …… テーブル間の結合条件を定義するFACTS         …… 集計の対象になる数値カラムを定義するDIMENSIONS    …… 切り口になるカラム（日付・カテゴリなど）を定義するMETRICS       …… FACTS を集計した業務指標を定義する```

## 2. メタデータを確認・自動生成するセマンティックビューを作る前に、テーブル自体のメタデータ（コメント）を確認します。AI はコメントも手がかりにするため、ここが空だと精度が落ちます。

In [ ]:
-- ==========================================================================-- 現在のテーブルコメントを確認-- ==========================================================================SELECT TABLE_NAME AS "テーブル",       COMMENT    AS "テーブルコメント"FROM SNOW_AM_DB.INFORMATION_SCHEMA.TABLESWHERE TABLE_SCHEMA = 'MARKET_INTELLIGENCE'  AND TABLE_TYPE = 'BASE TABLE'ORDER BY TABLE_NAME;

`setup.sql` でテーブルレベルのコメントは付けましたが、**カラムレベルのコメントは空**です。カラム数が多いテーブルにひとつずつ書くのは現実的ではありません。Snowflake には `AI_GENERATE_TABLE_DESC` というストアドプロシージャがあり、テーブル定義とサンプルデータから**説明文を自動生成**できます。| 引数 | 内容 ||---|---|| `table_name` | 対象テーブル（完全修飾名） || `describe_columns` | `TRUE` にするとカラムの説明も生成する || `use_table_data` | `TRUE` にすると実データをサンプリングして精度を上げる |> **注意**: これは**関数ではなくストアドプロシージャ**です。`SELECT` ではなく `CALL` で呼びます。

In [ ]:
-- ==========================================================================-- AI_GENERATE_TABLE_DESC でメタデータを自動生成-- ==========================================================================-- ⚠️ 実行に30秒程度かかりますCALL AI_GENERATE_TABLE_DESC(    'SNOW_AM_DB.MARKET_INTELLIGENCE.FACT_FUND_HOLDING',    {'describe_columns': TRUE, 'use_table_data': TRUE});

> **💡 自動生成と手書きの使い分け**>> 生成された説明を見ると、`WEIGHT_PCT` は「ファンドの総額に対して特定の保有銘柄が占める割合」と> 正しく解釈されています。データ型とサンプル値だけでここまで推測できるのは強力です。>> 一方で、出力は**英語**であり、「AUM」「構成比」といった**社内で使われる業務用語**は出てきません。>> 実務では次の使い分けが有効です。>> | 用途 | 手段 |> |---|---|> | カラム数が多いテーブルの一次的な説明付け | `AI_GENERATE_TABLE_DESC` で自動生成 → レビューして `ALTER TABLE ... SET COMMENT` |> | 業務指標の定義・業務用語の紐付け | セマンティックビューに**手で書く**（`METRICS` と `WITH SYNONYMS`） |>> 本ハンズオンでは後者、つまり業務用語を明示的に定義する方法を次のセクションで実践します。> 生成した説明をコメントとして保存する場合は> `ALTER TABLE ... MODIFY COLUMN <列> COMMENT '<説明>'` を実行します。

## 3. セマンティックビューを作成する作成方法は2つあります。| 方法 | 向いている場面 ||---|---|| **オプション1**: Snowsight の GUI ウィザード | 初めて作るとき。テーブル選択から関係性の推論までガイドされる || **オプション2**: SQL DDL | 定義をコード管理したいとき。レビューや再現が容易 |本ハンズオンでは時間の都合で**オプション2（SQL DDL）**で作成しますが、GUI の手順も記載しておきます。

### オプション1: Snowsight の GUI で作成する（参考）1. Snowsight のナビゲーションメニューから **AI と ML** » **Cortex Analyst** を開く2. **Create new** » **Create new Semantic View** を選ぶ3. 以下を設定する| 設定項目 | 値 ||---|---|| Database | `SNOW_AM_DB` || Schema | `MARKET_INTELLIGENCE` || Name | `PORTFOLIO_MARKET_SEMANTIC_VIEW` || Description | スノーアセットマネジメントのポートフォリオ・市場分析セマンティックビュー || Tables | `DIM_SECURITY`, `DIM_FUND`, `FACT_FUND_HOLDING`, `FACT_STOCK_PRICE_DAILY`, `FACT_FINANCIAL_METRICS`, `GOLD_COMPANY_NEWS_ANALYZED` || Columns | すべて選択 |4. 推論された関係性（Relationships）とメトリクスの提案をレビューして受け入れる5. **Save** をクリック> **💡 GUI 利用時の注意**>> - パフォーマンスのため、**10以上のテーブルを選択しない**こと> - **50列以上を選択しない**こと> - 自動提案された関係性は必ずレビューすること。誤った JOIN が提案されることがあります

### オプション2: SQL DDL で作成する以下が本ハンズオンで使うセマンティックビューの定義です。長いですが、構造は`TABLES` → `RELATIONSHIPS` → `FACTS` → `DIMENSIONS` → `METRICS` の5ブロックだけです。読むときのポイントを3つだけ挙げます。1. **`WITH SYNONYMS` に日本語と英語の両方を入れている** — 「AUM」でも「運用資産残高」でもヒットします2. **`xxx_RECORD AS 1` というダミーの FACT がある** — `COUNT()` メトリクスを定義するための定石です3. **`METRICS` に計算式を書ける** — 営業利益率のような比率指標もここで定義します

In [ ]:
-- ==========================================================================-- セマンティックビューの作成-- ==========================================================================CREATE OR REPLACE SEMANTIC VIEW SNOW_AM_DB.MARKET_INTELLIGENCE.PORTFOLIO_MARKET_SEMANTIC_VIEWTABLES (    securities AS SNOW_AM_DB.MARKET_INTELLIGENCE.DIM_SECURITY        PRIMARY KEY (TICKER)        WITH SYNONYMS = ('銘柄', '証券', '企業', 'security', 'stock', 'company')        COMMENT = '銘柄マスタ。米国大型株10銘柄の企業名・セクター・上場市場',    funds AS SNOW_AM_DB.MARKET_INTELLIGENCE.DIM_FUND        PRIMARY KEY (FUND_ID)        WITH SYNONYMS = ('ファンド', '投信', '投資信託', 'fund', 'portfolio')        COMMENT = 'ファンドマスタ。当社が運用する3ファンドの属性情報',    holdings AS SNOW_AM_DB.MARKET_INTELLIGENCE.FACT_FUND_HOLDING        PRIMARY KEY (FUND_ID, TICKER)        WITH SYNONYMS = ('保有明細', '保有', 'ポートフォリオ明細', 'holding', 'position')        COMMENT = 'ファンド保有明細。ファンド×銘柄の保有株数・保有時価・構成比',    prices AS SNOW_AM_DB.MARKET_INTELLIGENCE.FACT_STOCK_PRICE_DAILY        PRIMARY KEY (TICKER, TRADE_DATE)        WITH SYNONYMS = ('株価', '日次株価', '相場', 'price', 'stock price')        COMMENT = '日次株価。始値・高値・安値・終値・出来高',    financials AS SNOW_AM_DB.MARKET_INTELLIGENCE.FACT_FINANCIAL_METRICS        PRIMARY KEY (TICKER, PERIOD_END_DATE)        WITH SYNONYMS = ('財務諸表', '業績', '決算', 'financials', 'earnings')        COMMENT = 'SEC提出書類ベースの四半期財務諸表。売上・粗利・営業利益・純利益・EPS',    news AS SNOW_AM_DB.MARKET_INTELLIGENCE.GOLD_COMPANY_NEWS_ANALYZED        PRIMARY KEY (NEWS_ID)        WITH SYNONYMS = ('ニュース', '記事', '報道', 'news', 'article')        COMMENT = 'AI分析済み企業ニュース。センチメントとイベント種別を付与済み')RELATIONSHIPS (    holdings_to_funds       AS holdings   (FUND_ID) REFERENCES funds,    holdings_to_securities  AS holdings   (TICKER)  REFERENCES securities,    prices_to_securities    AS prices     (TICKER)  REFERENCES securities,    financials_to_securities AS financials (TICKER) REFERENCES securities,    news_to_securities      AS news       (TICKER)  REFERENCES securities)FACTS (    securities.SECURITY_RECORD AS 1 COMMENT = '銘柄レコード（件数カウント用）',    holdings.HOLDING_RECORD AS 1 COMMENT = '保有レコード（件数カウント用）',    holdings.MARKET_VALUE   AS MARKET_VALUE COMMENT = '保有時価（米ドル）',    holdings.SHARES         AS SHARES       COMMENT = '保有株数',    holdings.WEIGHT_PCT     AS WEIGHT_PCT   COMMENT = 'ファンド内構成比（％）',    prices.PRICE_RECORD AS 1 COMMENT = '株価レコード（件数カウント用）',    prices.CLOSE_PRICE  AS CLOSE_PRICE COMMENT = '終値（米ドル）',    prices.OPEN_PRICE   AS OPEN_PRICE  COMMENT = '始値（米ドル）',    prices.HIGH_PRICE   AS HIGH_PRICE  COMMENT = '高値（米ドル）',    prices.LOW_PRICE    AS LOW_PRICE   COMMENT = '安値（米ドル）',    prices.VOLUME       AS VOLUME      COMMENT = '出来高（株）',    financials.FINANCIAL_RECORD AS 1 COMMENT = '財務レコード（件数カウント用）',    financials.REVENUE          AS REVENUE          COMMENT = '売上高（米ドル）',    financials.GROSS_PROFIT     AS GROSS_PROFIT     COMMENT = '売上総利益（米ドル）',    financials.OPERATING_INCOME AS OPERATING_INCOME COMMENT = '営業利益（米ドル）',    financials.NET_INCOME       AS NET_INCOME       COMMENT = '純利益（米ドル）',    financials.EPS_DILUTED      AS EPS_DILUTED      COMMENT = '希薄化後1株当たり利益（米ドル）',    financials.RND_EXPENSE      AS RND_EXPENSE      COMMENT = '研究開発費（米ドル）',    news.NEWS_RECORD AS 1 COMMENT = 'ニュースレコード（件数カウント用）')DIMENSIONS (    securities.TICKER          AS TICKER        WITH SYNONYMS = ('ティッカー', '銘柄コード', 'symbol') COMMENT = 'ティッカーシンボル',    securities.COMPANY_NAME    AS COMPANY_NAME        WITH SYNONYMS = ('企業名', '会社名', 'company name') COMMENT = '企業名（英語）',    securities.COMPANY_NAME_JA AS COMPANY_NAME_JA        WITH SYNONYMS = ('企業名', '会社名', '日本語名') COMMENT = '企業名（日本語）',    securities.SECTOR          AS SECTOR        WITH SYNONYMS = ('セクター', '業種', 'industry', 'sector') COMMENT = 'セクター（日本語）',    securities.EXCHANGE        AS EXCHANGE        WITH SYNONYMS = ('取引所', '上場市場', 'exchange') COMMENT = '上場取引所',    funds.FUND_ID          AS FUND_ID        WITH SYNONYMS = ('ファンドID', 'fund id') COMMENT = 'ファンドID',    funds.FUND_NAME        AS FUND_NAME        WITH SYNONYMS = ('ファンド名', '投信名', 'fund name') COMMENT = 'ファンド名',    funds.FUND_TYPE        AS FUND_TYPE        WITH SYNONYMS = ('ファンド種別', 'fund type') COMMENT = 'ファンド種別',    funds.INVESTMENT_STYLE AS INVESTMENT_STYLE        WITH SYNONYMS = ('運用スタイル', '投資スタイル', 'style') COMMENT = '運用スタイル（グロース/コアなど）',    holdings.VALUATION_DATE AS VALUATION_DATE        WITH SYNONYMS = ('評価日', '基準日', 'valuation date') COMMENT = '保有時価の評価日',    prices.TRADE_DATE AS TRADE_DATE        WITH SYNONYMS = ('取引日', '営業日', '日付', 'date', 'trade date') COMMENT = '取引日',    financials.PERIOD_END_DATE      AS PERIOD_END_DATE        WITH SYNONYMS = ('期末日', '決算期末', 'period end') COMMENT = '会計期間の期末日',    financials.FISCAL_QUARTER_LABEL AS FISCAL_QUARTER_LABEL        WITH SYNONYMS = ('四半期', '会計四半期', 'quarter') COMMENT = '暦四半期ラベル（YYYY-Qn形式）',    news.PUBLISHED_AT   AS PUBLISHED_AT        WITH SYNONYMS = ('公開日', '配信日', 'published date') COMMENT = 'ニュース公開日',    news.SENTIMENT      AS SENTIMENT        WITH SYNONYMS = ('センチメント', '感情', '評判', 'sentiment') COMMENT = 'AI判定センチメント（ポジティブ/ネガティブ/ニュートラル/混在）',    news.EVENT_CATEGORY AS EVENT_CATEGORY        WITH SYNONYMS = ('イベント種別', 'ニュース種別', 'カテゴリ', 'category') COMMENT = 'AI分類したイベント種別',    news.HEADLINE       AS HEADLINE        WITH SYNONYMS = ('見出し', 'タイトル', 'headline') COMMENT = 'ニュース見出し',    news.SOURCE         AS SOURCE        WITH SYNONYMS = ('情報源', '出典', 'source') COMMENT = 'ニュース配信元')METRICS (    securities.SECURITY_COUNT AS COUNT(securities.SECURITY_RECORD)        WITH SYNONYMS = ('銘柄数', '企業数') COMMENT = '銘柄数',    holdings.TOTAL_MARKET_VALUE AS SUM(holdings.MARKET_VALUE)        WITH SYNONYMS = ('保有時価合計', 'AUM', '運用資産残高', '純資産') COMMENT = '保有時価の合計（米ドル）',    holdings.AVG_WEIGHT_PCT AS AVG(holdings.WEIGHT_PCT)        WITH SYNONYMS = ('平均構成比') COMMENT = 'ファンド内構成比の平均（％）',    holdings.MAX_WEIGHT_PCT AS MAX(holdings.WEIGHT_PCT)        WITH SYNONYMS = ('最大構成比') COMMENT = 'ファンド内構成比の最大値（％）',    holdings.TOTAL_SHARES AS SUM(holdings.SHARES)        WITH SYNONYMS = ('保有株数合計') COMMENT = '保有株数の合計',    holdings.HOLDING_COUNT AS COUNT(holdings.HOLDING_RECORD)        WITH SYNONYMS = ('保有銘柄数', 'ポジション数') COMMENT = '保有明細の件数',    prices.AVG_CLOSE_PRICE AS AVG(prices.CLOSE_PRICE)        WITH SYNONYMS = ('平均終値') COMMENT = '終値の平均（米ドル）',    prices.LATEST_CLOSE_PRICE AS MAX(prices.CLOSE_PRICE)        WITH SYNONYMS = ('最高終値', '期間高値') COMMENT = '期間内の終値の最大値（米ドル）',    prices.MIN_CLOSE_PRICE AS MIN(prices.CLOSE_PRICE)        WITH SYNONYMS = ('最低終値', '期間安値') COMMENT = '期間内の終値の最小値（米ドル）',    prices.TOTAL_VOLUME AS SUM(prices.VOLUME)        WITH SYNONYMS = ('出来高合計') COMMENT = '出来高の合計（株）',    financials.TOTAL_REVENUE AS SUM(financials.REVENUE)        WITH SYNONYMS = ('売上合計', '売上高') COMMENT = '売上高の合計（米ドル）',    financials.TOTAL_NET_INCOME AS SUM(financials.NET_INCOME)        WITH SYNONYMS = ('純利益合計') COMMENT = '純利益の合計（米ドル）',    financials.TOTAL_OPERATING_INCOME AS SUM(financials.OPERATING_INCOME)        WITH SYNONYMS = ('営業利益合計') COMMENT = '営業利益の合計（米ドル）',    financials.AVG_EPS_DILUTED AS AVG(financials.EPS_DILUTED)        WITH SYNONYMS = ('平均EPS', '平均希薄化後EPS') COMMENT = '希薄化後EPSの平均（米ドル）',    financials.AVG_OPERATING_MARGIN AS AVG(financials.OPERATING_INCOME) / NULLIF(AVG(financials.REVENUE), 0) * 100        WITH SYNONYMS = ('営業利益率', 'オペレーティングマージン') COMMENT = '営業利益率（％）',    financials.AVG_GROSS_MARGIN AS AVG(financials.GROSS_PROFIT) / NULLIF(AVG(financials.REVENUE), 0) * 100        WITH SYNONYMS = ('売上総利益率', '粗利率', 'グロスマージン') COMMENT = '売上総利益率（％）',    news.NEWS_COUNT AS COUNT(news.NEWS_RECORD)        WITH SYNONYMS = ('ニュース件数', '記事数') COMMENT = 'ニュースの件数')COMMENT = 'スノーアセットマネジメントのポートフォリオ・市場分析セマンティックビュー。自社ファンドの保有明細、米国株の日次株価、SEC提出書類ベースの四半期財務諸表、AI分析済み企業ニュースを統合し、自然言語での横断分析を可能にする。';

> **💡 定義のポイント解説**>> **ダミー FACT による件数カウント**>> ```sql> FACTS   ( holdings.HOLDING_RECORD AS 1 )> METRICS ( holdings.HOLDING_COUNT AS COUNT(holdings.HOLDING_RECORD) )> ```>> `METRICS` は `FACTS` を集計する形でしか書けません。単純な件数を数えたい場合は、> 常に `1` を返すダミーの FACT を定義してそれを `COUNT()` します。これは定石です。>> **比率指標の定義**>> ```sql> financials.AVG_OPERATING_MARGIN AS>     AVG(financials.OPERATING_INCOME) / NULLIF(AVG(financials.REVENUE), 0) * 100> ```>> `NULLIF(..., 0)` でゼロ除算を防いでいます。JPM のように売上が NULL の銘柄があるため、> この保護がないとエラーになります。>> **同義語は業務ヒアリングの成果物**>> ```sql> holdings.TOTAL_MARKET_VALUE AS SUM(holdings.MARKET_VALUE)>     WITH SYNONYMS = ('保有時価合計', 'AUM', '運用資産残高', '純資産')> ```>> ここに現場で実際に使われている呼び方を並べることが、Analyst の精度を最も左右します。> 技術的な作業ではなく、**業務部門との対話の成果**をここに書き込むイメージです。

## 4. 動作確認作成したセマンティックビューの定義を確認し、直接クエリしてみます。

In [ ]:
-- ==========================================================================-- セマンティックビューの定義を確認-- ==========================================================================DESCRIBE SEMANTIC VIEW PORTFOLIO_MARKET_SEMANTIC_VIEW;

### SEMANTIC_VIEW() 構文セマンティックビューは通常のビューとは異なり、専用の `SEMANTIC_VIEW()` 構文でクエリします。**使いたいメトリクスとディメンションを指定するだけ**で、必要な JOIN と `GROUP BY` はSnowflake が自動的に組み立てます。```sqlSELECT * FROM SEMANTIC_VIEW(    <セマンティックビュー名>    METRICS    <メトリクス名>, ...    DIMENSIONS <ディメンション名>, ...)```JOIN を書かなくても、`funds` テーブルのディメンションと `holdings` テーブルのメトリクスを組み合わせられることを確認してください。

In [ ]:
-- ==========================================================================-- Q1: ファンド別の運用資産残高と保有銘柄数-- ==========================================================================-- JOIN を1つも書いていないことに注目してくださいSELECT * FROM SEMANTIC_VIEW(    PORTFOLIO_MARKET_SEMANTIC_VIEW    METRICS    holdings.TOTAL_MARKET_VALUE, holdings.HOLDING_COUNT    DIMENSIONS funds.FUND_NAME)ORDER BY TOTAL_MARKET_VALUE DESC;

In [ ]:
-- ==========================================================================-- Q2: セクター別の保有時価（3テーブルを横断）-- ==========================================================================-- holdings（保有明細）のメトリクスを securities（銘柄マスタ）のセクターで切っていますSELECT * FROM SEMANTIC_VIEW(    PORTFOLIO_MARKET_SEMANTIC_VIEW    METRICS    holdings.TOTAL_MARKET_VALUE, holdings.HOLDING_COUNT    DIMENSIONS securities.SECTOR)ORDER BY TOTAL_MARKET_VALUE DESC;

In [ ]:
-- ==========================================================================-- Q3: 銘柄別の収益性指標-- ==========================================================================SELECT * FROM SEMANTIC_VIEW(    PORTFOLIO_MARKET_SEMANTIC_VIEW    METRICS    financials.AVG_OPERATING_MARGIN, financials.AVG_GROSS_MARGIN, financials.AVG_EPS_DILUTED    DIMENSIONS securities.TICKER, securities.COMPANY_NAME_JA)ORDER BY AVG_OPERATING_MARGIN DESC NULLS LAST;

In [ ]:
-- ==========================================================================-- Q4: 銘柄別のセンチメント別ニュース件数（非構造化データ由来）-- ==========================================================================-- Part 1 で AI が付与したセンチメントが、そのままディメンションとして使えますSELECT * FROM SEMANTIC_VIEW(    PORTFOLIO_MARKET_SEMANTIC_VIEW    METRICS    news.NEWS_COUNT    DIMENSIONS securities.COMPANY_NAME_JA, news.SENTIMENT)ORDER BY COMPANY_NAME_JA, NEWS_COUNT DESC;

> **💡 ここまでで達成したこと**>> Q3 では SEC 提出書類由来の財務データ、Q4 では PDF・CSV 由来の非構造化データを、> **同じセマンティックビューから同じ書き方で**取得できています。>> データの出自（Marketplace / PDF / CSV）を業務ユーザーが意識する必要はなくなりました。

## 5. Cortex Analyst で自然言語質問するここまでは「メトリクス名を知っている人」の書き方でした。Cortex Analyst を使うと、**メトリクス名を知らなくても自然言語で質問**できます。

### 5-1. Snowsight の UI から試す（推奨）1. Snowsight のナビゲーションメニューから **AI と ML** » **Cortex Analyst** を開く2. `SNOW_AM_DB.MARKET_INTELLIGENCE.PORTFOLIO_MARKET_SEMANTIC_VIEW` を選ぶ3. 以下の質問を順番に入れてみてください| # | 質問 | 確認したいこと ||---|---|---|| Q1 | 当社ファンドのAUM上位5銘柄を保有時価とセクターとあわせて教えて | 「AUM」という同義語が効いているか || Q2 | ファンドごとの運用資産残高と保有銘柄数を教えて | 「運用資産残高」でも同じ指標に当たるか || Q3 | NVDAの四半期売上の推移を教えて | 時系列の切り口が正しく選ばれるか || Q4 | セクター別の保有時価構成比を教えて | 構成比（全体に対する割合）を計算できるか || Q5 | ネガティブなニュースが最も多い銘柄はどこ | 非構造化データ由来の列を使えるか || Q6 | 営業利益率が最も高い銘柄を上位3つ教えて | 比率メトリクスの定義が使われるか |> **💡 見るべきポイント**>> Analyst は回答の前に **「This is our interpretation of your question」** として> 質問の解釈を返します。ここが自分の意図とずれていたら、同義語やメトリクスの定義を> 見直すサインです。生成された SQL も必ず開いて確認してください。

### 5-2. REST API から呼び出すCortex Analyst は REST API としても呼び出せます。アプリに組み込む場合はこちらを使います。Snowflake Notebook 内では `session.connection.rest.request` を使うと、トークンの管理をせずに API を呼べます。

In [ ]:
from snowflake.snowpark.context import get_active_sessionsession = get_active_session()SEMANTIC_VIEW = "SNOW_AM_DB.MARKET_INTELLIGENCE.PORTFOLIO_MARKET_SEMANTIC_VIEW"def ask_cortex_analyst(question: str, semantic_view: str = SEMANTIC_VIEW) -> dict:    """Cortex Analyst に質問を投げ、レスポンス全体を返す"""    body = {        "messages": [            {"role": "user", "content": [{"type": "text", "text": question}]}        ],        "semantic_view": semantic_view,    }    return session.connection.rest.request(        url="/api/v2/cortex/analyst/message",        body=body,        method="post",        client="rest",        timeout=120,    )def ask_and_run(question: str) -> None:    """質問を投げ、解釈・生成SQL・実行結果をまとめて表示する"""    print("=" * 78)    print("質問:", question)    response = ask_cortex_analyst(question)    generated_sql = None    for part in response["message"]["content"]:        if part["type"] == "text":            print("\n[Analyst の解釈]")            print(part["text"])        elif part["type"] == "sql":            generated_sql = part["statement"]    if generated_sql is None:        print("\nSQL が生成されませんでした。質問を具体的にしてみてください。")        return    print("\n[生成された SQL]")    print(generated_sql)    # 末尾のセミコロンを外してから実行します    print("\n[実行結果]")    df = session.sql(generated_sql.strip().rstrip(";")).to_pandas()    display(df.head(10))print("ヘルパ関数を定義しました")

In [ ]:
# まずは1問だけ試してみますask_and_run("当社ファンドのAUM上位5銘柄を保有時価とセクターとあわせて教えて")

In [ ]:
# 残りの質問をまとめて実行します（1問あたり10〜20秒かかります）QUESTIONS = [    "ファンドごとの運用資産残高と保有銘柄数を教えて",    "NVDAの四半期売上の推移を教えて",    "セクター別の保有時価構成比を教えて",    "ネガティブなニュースが最も多い銘柄はどこ",    "営業利益率が最も高い銘柄を上位3つ教えて",]for question in QUESTIONS:    ask_and_run(question)

> **💡 生成された SQL の読み方**>> Analyst が生成する SQL には2つのパターンがあります。どちらも正しい結果を返します。>> **パターン1: `SEMANTIC_VIEW()` 構文を使う**>> ```sql> SELECT * FROM SEMANTIC_VIEW(>     SNOW_AM_DB.MARKET_INTELLIGENCE.PORTFOLIO_MARKET_SEMANTIC_VIEW>     METRICS total_market_value>     DIMENSIONS securities.ticker, securities.sector> ) ORDER BY total_market_value DESC LIMIT 5> ```>> セマンティックビューで定義したメトリクスをそのまま使うため、**定義との一貫性が保証されます**。>> **パターン2: 元テーブルを直接参照する**>> ```sql> WITH __financials AS (SELECT ... FROM FACT_FINANCIAL_METRICS)> SELECT f.fiscal_quarter_label, SUM(f.revenue) ... GROUP BY ...> ```>> 定義済みメトリクスで表現できない集計（銘柄で絞った時系列など）ではこちらになります。> このときも、**どのテーブルをどう結合するかはセマンティックビューの定義に従っています**。>> いずれの場合も末尾に `-- Generated by Cortex Analyst (request_id: ...)` が付きます。> この `request_id` は問い合わせやデバッグの際に使えるため、控えておくと役立ちます。

## まとめ### 作成したオブジェクト| オブジェクト | 内容 ||---|---|| `PORTFOLIO_MARKET_SEMANTIC_VIEW` | 6テーブル・18メトリクスを統合したセマンティックビュー |### このパートで押さえたポイント1. **Marketplace の生データは縦持ち**。横持ちに整えないと AI は静かに間違った SQL を書く2. **セマンティックビューが加えるのは「結合条件」「指標定義」「業務用語」の3つ**3. **`WITH SYNONYMS` が精度を最も左右する**。技術作業ではなく業務部門との対話の成果物4. **`xxx_RECORD AS 1` のダミー FACT** で `COUNT()` メトリクスを定義するのが定石5. **`SEMANTIC_VIEW()` 構文なら JOIN を書かなくてよい**。必要な結合は定義から自動生成される6. **構造化データと非構造化データ由来の列を同じ書き方で扱える**### 次のステップ`part3_cortex_agent.ipynb` に進み、Cortex Search を追加して「数値を集計する Analyst」と「文書を検索する Search」を統合した Cortex Agent を作成します。そして Snowflake CoWork から実際に使ってみます。